# 04 — Feature extraction (first 24h)

For each ICU stay in the cohort, I extract a set of features from the first 24 hours of the stay: vitals from `chartevents` and selected labs from `labevents`. For each variable I take the min, max, and mean over the window — a simple but standard summary that's good enough for a baseline.

## Picking which variables

For a first pass I keep the list short and clinical-textbook (the vitals every ICU patient gets, and the labs most commonly used in severity scores). This is not an exhaustive feature set — it's the readable baseline that's been used in many MIMIC mortality papers.

## What I am NOT doing here

- No proper time-series modeling (no last-value-carried-forward, no trends, no early-warning patterns). Just min/max/mean.
- No careful unit reconciliation or outlier clipping. I rely on `valuenum` being clean enough for the baseline; a production pipeline would do more.
- Demographics are NOT used as model inputs (except age). Gender, race, and insurance are held aside as grouping variables for the fairness audit.

## Setup

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np

DATA_DIR = Path("../data")
HOSP = DATA_DIR / "hosp"
ICU = DATA_DIR / "icu"
DERIVED = DATA_DIR / "derived"

con = duckdb.connect()

## Load cohort

In [2]:
cohort = pd.read_csv(
    DERIVED / "cohort.csv",
    parse_dates=["intime", "outtime"],
)
print(f"cohort: {len(cohort)} stays / {cohort['subject_id'].nunique()} patients")
cohort.head()

cohort: 85 stays / 85 patients


,subject_id,hadm_id,stay_id,intime,outtime,los,first_careunit,gender,anchor_age,race,insurance,marital_status,hospital_expire_flag
0,10001217,24597018,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,Surgical Intensive Care Unit (SICU),F,55,WHITE,Other,MARRIED,0
1,10001725,25563031,31205490,2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588,Medical/Surgical Intensive Care Unit (MICU/SICU),F,46,WHITE,Other,MARRIED,0
2,10002428,28662225,33987268,2156-04-12 16:24:18,2156-04-17 15:57:08,4.981134,Medical Intensive Care Unit (MICU),F,80,WHITE,Medicare,WIDOWED,0
3,10002495,24982426,36753294,2141-05-22 20:18:01,2141-05-27 22:24:02,5.087512,Coronary Care Unit (CCU),M,81,UNKNOWN,Medicare,MARRIED,0
4,10002930,25696644,37049133,2196-04-14 13:40:00,2196-04-15 16:54:44,1.135231,Medical Intensive Care Unit (MICU),F,48,BLACK/AFRICAN AMERICAN,Medicare,SINGLE,0


## Variable mapping

MIMIC-IV identifies each measurement by `itemid`, not by clinical name. I hand-curate the itemids I want (cross-referenced with the official `d_items` and `d_labitems` dictionaries). When a clinical concept has multiple itemids (e.g., invasive vs non-invasive blood pressure), I include both and let the aggregation collapse them into the same column.

In [3]:
# Vitals from chartevents (icu module)
vital_itemids = {
    220045: "heart_rate",
    220179: "sbp",         # NIBP systolic
    220050: "sbp",         # Arterial systolic
    220180: "dbp",         # NIBP diastolic
    220051: "dbp",         # Arterial diastolic
    220210: "resp_rate",
    220277: "spo2",
    223762: "temp_c",      # temperature, Celsius
}

# Labs from labevents (hosp module)
lab_itemids = {
    50912: "creatinine",
    51006: "bun",
    50931: "glucose",
    50983: "sodium",
    50971: "potassium",
    51222: "hemoglobin",
    51301: "wbc",
    50813: "lactate",
}

print(f"{len(set(vital_itemids.values()))} unique vital concepts")
print(f"{len(set(lab_itemids.values()))} unique lab concepts")

6 unique vital concepts
8 unique lab concepts


## Extract vitals from `chartevents`

DuckDB pushes the time + itemid filter into the CSV scan, so I don't have to load the full `chartevents` into memory. Same approach will work on full MIMIC-IV later.

In [4]:
# Register the cohort DataFrame so DuckDB can JOIN against it
con.register("cohort", cohort)

vital_id_list = ", ".join(str(i) for i in vital_itemids)
chart_path = str(ICU / "chartevents.csv.gz")

vitals_long = con.execute(f"""
    SELECT
        co.stay_id,
        c.itemid,
        MIN(c.valuenum) AS min_val,
        MAX(c.valuenum) AS max_val,
        AVG(c.valuenum) AS mean_val,
        COUNT(c.valuenum) AS n
    FROM '{chart_path}' AS c
    INNER JOIN cohort AS co ON c.stay_id = co.stay_id
    WHERE c.itemid IN ({vital_id_list})
      AND c.valuenum IS NOT NULL
      AND c.charttime >= co.intime
      AND c.charttime < co.intime + INTERVAL 24 HOUR
    GROUP BY co.stay_id, c.itemid
""").df()

print(f"vitals long: {vitals_long.shape}")
vitals_long.head()

vitals long: (517, 6)


,stay_id,itemid,min_val,max_val,mean_val,n
0,32604416,220051,35.0,67.0,44.536585,41
1,32604416,223762,36.2,37.3,36.867500,40
2,32604416,220050,87.0,172.0,124.317073,41
3,32506122,220045,57.0,80.0,65.692308,26
4,39711498,220045,29.0,127.0,110.960000,25


## Pivot vitals to one row per stay

In [5]:
# Map itemid -> concept name
vitals_long["feature"] = vitals_long["itemid"].map(vital_itemids)

# Some itemids map to the same concept (e.g., NIBP + Arterial sbp).
# Collapse them by re-aggregating across (stay_id, feature).
vitals_collapsed = (
    vitals_long
    .groupby(["stay_id", "feature"], as_index=False)
    .agg(min_val=("min_val", "min"),
         max_val=("max_val", "max"),
         mean_val=("mean_val", "mean"))
)

vitals_wide = vitals_collapsed.pivot(
    index="stay_id",
    columns="feature",
    values=["min_val", "max_val", "mean_val"],
)
vitals_wide.columns = [f"{stat.replace('_val', '')}_{feat}" for stat, feat in vitals_wide.columns]
vitals_wide = vitals_wide.reset_index()
print(f"vitals wide: {vitals_wide.shape}")
vitals_wide.head()

vitals wide: (85, 19)


,stay_id,min_dbp,min_heart_rate,min_resp_rate,min_sbp,min_spo2,min_temp_c,max_dbp,max_heart_rate,max_resp_rate,max_sbp,max_spo2,max_temp_c,mean_dbp,mean_heart_rate,mean_resp_rate,mean_sbp,mean_spo2,mean_temp_c
0,30057454,47.0,100.0,13.0,58.0,89.0,NaN,77.0,114.0,25.0,113.0,96.0,NaN,59.333333,108.533333,18.266667,87.300000,92.366667,NaN
1,30101877,49.0,79.0,16.0,102.0,100.0,NaN,86.0,123.0,25.0,166.0,100.0,NaN,69.750000,94.875000,20.541667,137.916667,100.000000,NaN
2,30458995,45.0,60.0,14.0,104.0,89.0,NaN,120.0,92.0,24.0,157.0,99.0,NaN,66.428571,81.720000,17.600000,129.940476,97.360000,NaN
3,30585761,35.0,60.0,10.0,111.0,90.0,NaN,59.0,81.0,24.0,143.0,98.0,NaN,51.652174,70.958333,19.375000,132.717391,95.181818,NaN
4,30665396,51.0,70.0,10.0,90.0,95.0,36.6,81.0,105.0,36.0,135.0,100.0,36.6,61.407407,91.346154,20.280000,108.518519,98.650000,36.6


## Extract labs from `labevents`

Same pattern as vitals, but joined on `hadm_id` — labs are recorded at admission level, not per ICU stay.

In [6]:
lab_id_list = ", ".join(str(i) for i in lab_itemids)
lab_path = str(HOSP / "labevents.csv.gz")

labs_long = con.execute(f"""
    SELECT
        co.stay_id,
        l.itemid,
        MIN(l.valuenum) AS min_val,
        MAX(l.valuenum) AS max_val,
        AVG(l.valuenum) AS mean_val,
        COUNT(l.valuenum) AS n
    FROM '{lab_path}' AS l
    INNER JOIN cohort AS co ON l.hadm_id = co.hadm_id
    WHERE l.itemid IN ({lab_id_list})
      AND l.valuenum IS NOT NULL
      AND l.charttime >= co.intime
      AND l.charttime < co.intime + INTERVAL 24 HOUR
    GROUP BY co.stay_id, l.itemid
""").df()

print(f"labs long: {labs_long.shape}")
labs_long.head()

labs long: (651, 6)


,stay_id,itemid,min_val,max_val,mean_val,n
0,33683112,50983,137.0,138.0,137.50,2
1,34629895,51006,16.0,17.0,16.50,2
2,38430513,50813,5.9,6.3,6.10,2
3,32128372,50931,113.0,148.0,135.20,5
4,35258379,50971,3.8,4.5,4.15,2


In [7]:
labs_long["feature"] = labs_long["itemid"].map(lab_itemids)

labs_wide = labs_long.pivot(
    index="stay_id",
    columns="feature",
    values=["min_val", "max_val", "mean_val"],
)
labs_wide.columns = [f"{stat.replace('_val', '')}_{feat}" for stat, feat in labs_wide.columns]
labs_wide = labs_wide.reset_index()
print(f"labs wide: {labs_wide.shape}")
labs_wide.head()

labs wide: (85, 25)


,stay_id,min_bun,min_creatinine,min_glucose,min_hemoglobin,min_lactate,min_potassium,min_sodium,min_wbc,max_bun,...,max_sodium,max_wbc,mean_bun,mean_creatinine,mean_glucose,mean_hemoglobin,mean_lactate,mean_potassium,mean_sodium,mean_wbc
0,30057454,39.0,1.7,119.0,13.1,0.6,3.3,135.0,12.7,45.0,...,141.0,17.8,42.000000,1.750000,134.000000,13.750000,0.600000,3.825000,137.5,15.250000
1,30101877,18.0,0.7,153.0,9.9,NaN,4.5,135.0,19.8,26.0,...,137.0,22.1,22.000000,0.900000,184.500000,10.150000,NaN,4.700000,136.0,20.950000
2,30458995,10.0,0.6,100.0,11.2,1.1,3.5,134.0,12.7,11.0,...,140.0,21.5,10.666667,0.616667,113.833333,11.457143,1.185714,3.883333,136.5,18.028571
3,30585761,37.0,1.3,78.0,11.0,NaN,3.8,139.0,10.5,37.0,...,139.0,10.5,37.000000,1.300000,78.000000,11.000000,NaN,3.800000,139.0,10.500000
4,30665396,12.0,1.0,137.0,8.5,1.4,4.6,134.0,8.8,14.0,...,139.0,16.2,13.000000,1.000000,137.000000,10.433333,2.775000,4.750000,136.5,13.533333


## Combine into the feature matrix

I keep cohort metadata (identifiers, demographics, outcome) alongside the extracted features. Demographics + outcome are NOT model inputs — they're kept here for the fairness audit later.

In [8]:
metadata_cols = [
    "stay_id", "subject_id", "hadm_id",
    "anchor_age", "gender", "race", "insurance",
    "hospital_expire_flag",
]

features = (
    cohort[metadata_cols]
    .merge(vitals_wide, on="stay_id", how="left")
    .merge(labs_wide, on="stay_id", how="left")
)

print(f"feature matrix: {features.shape}")
features.head()

feature matrix: (85, 50)


,stay_id,subject_id,hadm_id,anchor_age,gender,race,insurance,hospital_expire_flag,min_dbp,min_heart_rate,...,max_sodium,max_wbc,mean_bun,mean_creatinine,mean_glucose,mean_hemoglobin,mean_lactate,mean_potassium,mean_sodium,mean_wbc
0,37067082,10001217,24597018,55,F,WHITE,Other,0,67.0,78.0,...,138.0,19.0,9.000000,0.400000,113.0,11.200000,NaN,3.600000,138.000000,19.000000
1,31205490,10001725,25563031,46,F,WHITE,Other,0,45.0,55.0,...,140.0,20.1,17.000000,0.800000,149.0,13.250000,NaN,3.700000,139.000000,18.550000
2,33987268,10002428,28662225,80,F,WHITE,Medicare,0,19.0,100.0,...,133.0,27.9,13.500000,0.850000,123.5,9.750000,1.825000,3.833333,130.000000,25.150000
3,36753294,10002495,24982426,81,M,UNKNOWN,Medicare,0,40.0,83.0,...,131.0,36.8,33.666667,1.533333,290.0,13.766667,3.666667,4.033333,129.666667,31.533333
4,37049133,10002930,25696644,48,F,BLACK/AFRICAN AMERICAN,Medicare,0,48.0,76.0,...,136.0,2.4,15.000000,0.600000,107.0,7.700000,NaN,3.400000,136.000000,2.400000


## Missing data

With a small cohort and many measurements, some features will have low coverage. I check the missing rate per feature before deciding what to do.

In [9]:
feature_cols = [c for c in features.columns if c not in metadata_cols]
missing_rates = features[feature_cols].isna().mean().sort_values(ascending=False)
missing_rates

max_temp_c         0.858824
min_temp_c         0.858824
mean_temp_c        0.858824
max_lactate        0.341176
min_lactate        0.341176
mean_lactate       0.341176
mean_hemoglobin    0.000000
min_sodium         0.000000
min_wbc            0.000000
max_bun            0.000000
max_creatinine     0.000000
mean_sodium        0.000000
max_glucose        0.000000
max_hemoglobin     0.000000
mean_potassium     0.000000
max_potassium      0.000000
max_sodium         0.000000
min_potassium      0.000000
max_wbc            0.000000
mean_bun           0.000000
mean_creatinine    0.000000
mean_glucose       0.000000
min_dbp            0.000000
min_hemoglobin     0.000000
min_heart_rate     0.000000
min_glucose        0.000000
min_resp_rate      0.000000
min_sbp            0.000000
min_spo2           0.000000
max_dbp            0.000000
max_heart_rate     0.000000
max_resp_rate      0.000000
max_sbp            0.000000
max_spo2           0.000000
mean_dbp           0.000000
mean_heart_rate    0

## Simple imputation

For the baseline I impute missing numeric features with the median across the cohort. This is the most naive defensible choice — production pipelines do better (per-stay LOCF, model-based imputation, explicit missingness indicators). I log what got imputed for transparency.

In [10]:
features_imputed = features.copy()
imputed_log = {}

for col in feature_cols:
    if features_imputed[col].isna().any():
        median = features_imputed[col].median()
        n_missing = features_imputed[col].isna().sum()
        features_imputed[col] = features_imputed[col].fillna(median)
        imputed_log[col] = {"median": float(median), "n_missing": int(n_missing)}

print(f"imputed {len(imputed_log)} columns")
pd.DataFrame(imputed_log).T.head(10)

imputed 6 columns


,median,n_missing
min_temp_c,35.550000,73.0
max_temp_c,37.200000,73.0
mean_temp_c,36.417033,73.0
min_lactate,1.600000,29.0
max_lactate,2.550000,29.0
mean_lactate,2.125000,29.0


## Encode gender

Binary categorical, kept available for downstream slicing (not as a model input here).

In [11]:
features_imputed["gender_f"] = (features_imputed["gender"] == "F").astype(int)
features_imputed[["gender", "gender_f"]].head()

,gender,gender_f
0,F,1
1,F,1
2,F,1
3,M,0
4,F,1


## Save the feature matrix

In [12]:
out_path = DERIVED / "features.csv"
features_imputed.to_csv(out_path, index=False)
print(f"saved to {out_path}")
print(f"shape: {features_imputed.shape}")
print(f"file size: {out_path.stat().st_size / 1024:.1f} KB")

saved to ../data/derived/features.csv
shape: (85, 51)
file size: 29.6 KB


## Takeaways

- I have a flat feature matrix: one row per ICU stay, ~24 clinical features (8 concepts × 3 statistics) + identifiers + demographics + outcome.
- Coverage is uneven; median imputation is honest but crude.
- The cohort is small enough that some features may end up nearly constant after imputation — something to keep in mind when reading the model coefficients later.

Next (notebook 05): split train/test, train a logistic regression baseline on the clinical features (excluding demographics), and look at overall performance.